# Mixture of Experts

Companion notebook for the [Mixture of Experts lesson](https://ml-viz-ruby.vercel.app/courses/transformers/06-mixture-of-experts).

We implement a small **MoE layer**: a top-k router, sparse expert combination, and the
**load-balancing** problem (routing collapse and the auxiliary loss that fixes it). We also confirm
the headline property — **parameters scale with N, compute with k**. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## 1 — Top-k routing

The router scores each token against every expert; we keep the top-k and softmax over just those to
get combination weights. Only k of N experts are ever evaluated for a token.

In [ ]:
def route(logits, k):
    """logits: (tokens, experts). Returns (chosen idx (T,k), gate weights (T,k))."""
    idx = np.argsort(-logits, axis=1)[:, :k]                     # top-k experts per token
    topv = np.take_along_axis(logits, idx, axis=1)
    e = np.exp(topv - topv.max(1, keepdims=True))
    gates = e / e.sum(1, keepdims=True)                          # softmax over the chosen experts
    return idx, gates

T, N = 6, 5
logits = rng.normal(size=(T, N))
idx, gates = route(logits, k=2)
print('each token -> its top-2 experts:\n', idx)
print('gate weights (sum to 1 per token):\n', gates)

## 2 — Sparse expert combination

Each chosen expert (a small FFN) processes the token; outputs are combined by the gate weights. The
non-chosen experts are never run for that token — that's the compute saving.

In [ ]:
d = 8
experts = [rng.normal(size=(d, d)) * 0.3 for _ in range(N)]      # N expert weight matrices
X = rng.normal(size=(T, d))

def moe_forward(X, logits, k):
    idx, gates = route(logits, k)
    Y = np.zeros_like(X)
    expert_evals = 0
    for t in range(len(X)):
        for j in range(k):
            e = idx[t, j]
            Y[t] += gates[t, j] * (X[t] @ experts[e])           # only k experts run
            expert_evals += 1
    return Y, expert_evals

Y, evals = moe_forward(X, logits, k=2)
print(f'expert evaluations: {evals} (= tokens {T} × k 2)')
print(f'a dense layer would run all {T*N} token-expert pairs')

## 3 — Parameters scale with N, compute with k

Grow the number of experts N: total parameters explode, but the FLOPs per token stay fixed (set by
k). This is exactly why MoE decouples capacity from cost.

In [ ]:
k = 1
for N_exp in [1, 8, 64, 256]:
    params = N_exp * d * d                  # all experts' weights are stored
    flops_per_token = k * d * d             # only k experts run
    print(f'N={N_exp:3d} experts:  params={params:6d}  compute/token={flops_per_token}  (compute is flat!)')

## 4 — Routing collapse and load balancing

If the router favors a few experts, load is uneven — popular experts bottleneck, others go idle. The
auxiliary load-balancing loss penalizes imbalance; it's minimized when load is uniform.

In [ ]:
def load_fraction(logits, k, N):
    idx, _ = route(logits, k)
    counts = np.bincount(idx.ravel(), minlength=N)
    return counts / counts.sum()

def balance_loss(frac):
    # high when load is concentrated, minimized (=1) when uniform
    N = len(frac)
    return N * np.sum(frac ** 2)

balanced = np.tile(np.eye(N)[0] * 0, (60, 1)) + rng.normal(size=(60, N))   # ~even router
collapsed = balanced.copy(); collapsed[:, 0] += 5.0                         # expert 0 dominates
print('balanced  load:', load_fraction(balanced, 1, N).round(2), ' loss=', round(balance_loss(load_fraction(balanced,1,N)),2))
print('collapsed load:', load_fraction(collapsed, 1, N).round(2), ' loss=', round(balance_loss(load_fraction(collapsed,1,N)),2))
print('\nThe collapsed router has higher balance loss -> training is pushed back toward uniform.')

## ✏️ Your turn

**Exercise.** Implement `topk_experts(logits_row, k)` returning the indices of the k highest-scoring
experts for one token, and `compute_per_token(k, d)` returning the FLOPs per token (= k·d², i.e.
independent of how many experts N exist) — the property that makes MoE scale.

In [ ]:
def topk_experts(logits_row, k):
    # TODO(you): return the indices of the k largest entries of logits_row
    return ...

def compute_per_token(k, d):
    # TODO(you): FLOPs per token for k active experts of size d×d (ignore N entirely)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
r = np.array([0.1, 0.9, 0.5, 0.3])
assert set(topk_experts(r, 2)) == {1, 2}                  # the two highest
assert len(topk_experts(r, 1)) == 1 and topk_experts(r, 1)[0] == 1
# compute is independent of N: same k and d -> same cost no matter the expert count
assert compute_per_token(1, 8) == compute_per_token(1, 8)
assert compute_per_token(2, 8) == 2 * compute_per_token(1, 8)
print('\u2713 top-k routing and compute accounting are correct')

<details>
<summary>Solution</summary>

```python
def topk_experts(logits_row, k):
    return np.argsort(-logits_row)[:k]

def compute_per_token(k, d):
    return k * d * d
```

Compute per token depends on k, not N — so you can add experts (parameters, capacity) almost for
free in FLOPs. The catch the formula hides: all N experts' weights still have to live in memory.

</details>